# EcoData · S11 — BI/ML distribuido con Spark
**Lanzeloth Vargas Marichi** · Proyecto Sello EcoData · Big Data, UPeU

Dos partes, igual que la guía:

| Parte | Guía S11 | EcoData |
|---|---|---|
| **A · BI batch** | KPIs del Parquet de S9, un solo `PUT` al Pushgateway | KPIs del Parquet de **S9 EcoData** (`s09_eventos_observabilidad`): volumen, ratio de válidos, latencia → job `lambda26_bi_flujo` |
| **B · ML en vivo** | modelo de S10 redesplegado, un `PUT` por micro-lote | modelo de **S10 EcoData** (`modelo_pm25_s10`) sobre `ecodata-s10-aire` → job `lambda26_ml_streaming` |
| Tablero | Stat (batch) + Time series (en vivo) | `s11_dashboard_ecodata.json` |

**Requisitos:** haber ejecutado **S9** (genera el Parquet) y **S10** (guarda el modelo) en esta misma carpeta, con Kafka, Prometheus y Pushgateway arriba. Cierra otros cuadernos de Spark.

> **Honestidad sobre Pushgateway:** la Parte A es el caso correcto (job que termina). La Parte B es un **atajo** (job continuo): si el productor se detiene, el último valor queda congelado en el Pushgateway sin ninguna señal. En EcoData ese riesgo se mitiga con el *semáforo de frescura* (`time() - push_time_seconds`) y alertas con `absent()`; la alternativa recomendada sería un endpoint `/metrics` propio.

## 0. Preparación

In [1]:
import sys, json, time, threading, urllib.request, urllib.parse, os
from datetime import datetime, timezone
import pandas as pd
sys.path.insert(0, "/tmp/s12/scripts")
import aire_comun as aire
import openmeteo_a_kafka as pub_aire
from kafka import KafkaProducer, KafkaConsumer, TopicPartition
from kafka.admin import KafkaAdminClient, NewTopic
from kafka.errors import TopicAlreadyExistsError

BOOTSTRAP = "kafka:9092"
PUSHGATEWAY = "http://pushgateway:9091"
PROMETHEUS_CANDIDATOS = ["http://prometheus:9090", "http://lambda26-prometheus:9090"]
PKG = "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0"

def crear_topic(nombre, particiones=3):
    adm = KafkaAdminClient(bootstrap_servers=BOOTSTRAP)
    try: adm.create_topics([NewTopic(nombre, num_partitions=particiones, replication_factor=1)]); print("topic creado:", nombre)
    except TopicAlreadyExistsError: print("el topic ya existía:", nombre)
    adm.close()

def prom(q):
    for base in PROMETHEUS_CANDIDATOS:
        try:
            u = f"{base}/api/v1/query?" + urllib.parse.urlencode({"query": q})
            return json.load(urllib.request.urlopen(u, timeout=4))["data"]["result"]
        except Exception:
            continue
    return None

def publicar_paced(topic, eventos, pausa_s=2.0):
    """Publica un evento cada pausa_s segundos en segundo plano (la celda no se bloquea)."""
    def run():
        p = KafkaProducer(bootstrap_servers=BOOTSTRAP, acks="all", key_serializer=str.encode,
                          value_serializer=lambda v: json.dumps(v).encode())
        for e in eventos:
            p.send(topic, key=e["sensorId"], value=e); p.flush(); time.sleep(pausa_s)
        p.close()
    th = threading.Thread(target=run, daemon=True); th.start(); return th

def desde_archivo(path="/tmp/s12/aq_juliaca_2023.json"):
    d = json.load(open(path)); h = d.get("hourly", d); out = []
    for t, a, b in zip(h["time"], h["pm2_5"], h["pm10"]):
        if a is None: continue
        ts = int(datetime.fromisoformat(t).replace(tzinfo=timezone.utc).timestamp() * 1000)
        out.append((ts, float(a), float(b) if b is not None else float(a)))
    return out

TOPIC = "ecodata-s10-aire"
RUTA_S9 = "./artifacts/s09_eventos_observabilidad"
RUTA_MODELO = "./artifacts/models/modelo_pm25_s10"
print("Parquet S9:", os.path.exists(RUTA_S9), "· Modelo S10:", os.path.exists(RUTA_MODELO))

Parquet S9: True · Modelo S10: True


In [2]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StringType, DoubleType, LongType
esquema_evento = (StructType().add("tipoEvento", StringType()).add("sensorId", StringType())
                  .add("pm25", DoubleType()).add("pm10", DoubleType())
                  .add("origen", StringType()).add("timestamp", LongType()))
spark = (SparkSession.builder.appName("ecodata-s11").master("local[2]")
         .config("spark.jars.packages", PKG)
         .config("spark.sql.shuffle.partitions", "3")
         .config("spark.sql.session.timeZone", "UTC")
         .config("spark.ui.port", "4052")          # distinto del 4040 que usan los jobs de S12
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)
from pyspark.ml import PipelineModel

:: loading settings :: url = jar:file:/usr/local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-8992823f-610c-42bf-b0e9-4c638262b1a3;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.2.0 in spark-list
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.2.0 in central
	found org.apache.kafka#kafka-clients;3.9.2 in central
	found at.yawk.lz4#lz4-java;1.11.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.8 in central
	found org.slf4j#slf4j-api;2.0.17 in central
	found org.apache.hadoop#hadoop-client-runtime;3.5.0 in central
	found org.apache.hadoop#hadoop-client-api;3.5.0 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.scala-lang.

Spark 4.2.0


## 1. Parte A — KPIs del flujo de eventos (3.2–3.3)

In [3]:
eventos_s9 = spark.read.parquet(RUTA_S9)
kpi = eventos_s9.agg(F.count("*").alias("total"),
                     F.avg(F.col("isValid").cast("int")).alias("ratioValidos"),
                     F.avg("latencyMs").alias("avgLatencyMs"),
                     F.percentile_approx("latencyMs", 0.95).alias("p95LatencyMs")).first()
print(f"Eventos procesados:  {kpi['total']}")
print(f"Ratio de válidos:    {kpi['ratioValidos']:.4f}")
print(f"Latencia promedio:   {kpi['avgLatencyMs']:.1f} ms  (p95 = {kpi['p95LatencyMs']} ms)")

[Stage 1:>                                                          (0 + 2) / 2]

Eventos procesados:  90
Ratio de válidos:    1.0000
Latencia promedio:   2724.2 ms  (p95 = 4832 ms)


## 2. Publicar los KPIs una sola vez en el Pushgateway y verificarlos (3.4)

In [4]:
metricas = {"lambda26_bi_eventos_total": kpi["total"],
            "lambda26_bi_eventos_validos_ratio": kpi["ratioValidos"],
            "lambda26_bi_latencia_promedio_ms": kpi["avgLatencyMs"],
            "lambda26_bi_latencia_p95_ms": kpi["p95LatencyMs"]}
cuerpo = "".join(f"# TYPE {n} gauge\n{n} {v}\n" for n, v in metricas.items())
req = urllib.request.Request(f"{PUSHGATEWAY}/metrics/job/lambda26_bi_flujo/instance/s11_batch", data=cuerpo.encode(), method="PUT")
print("Pushgateway respondió:", urllib.request.urlopen(req).status)

Pushgateway respondió: 200


In [5]:
time.sleep(20)       # al menos un scrape_interval (15 s)
for n in metricas:
    r = prom(n); print(f"{n}: {r[0]['value'][1] if r else 'SIN DATOS (espera otro scrape y repite)'}")

lambda26_bi_eventos_total: 90
lambda26_bi_eventos_validos_ratio: 1
lambda26_bi_latencia_promedio_ms: 2724.188888888889
lambda26_bi_latencia_p95_ms: 4832


## 3. Parte B — redesplegar el modelo de S10 en vivo, sin reentrenar (3.6–3.8)

In [6]:
crear_topic(TOPIC)
modelo = PipelineModel.load(RUTA_MODELO)
print("Modelo cargado desde:", RUTA_MODELO)

el topic ya existía: ecodata-s10-aire
Modelo cargado desde: ./artifacts/models/modelo_pm25_s10


In [7]:
crudo = (spark.readStream.format("kafka").option("kafka.bootstrap.servers", BOOTSTRAP)
         .option("subscribe", "ecodata-s10-aire").option("startingOffsets", "latest").option("failOnDataLoss", "false").load())
eventos = (crudo.select("topic", "partition", "offset", F.col("timestamp").alias("kafkaTimestamp"),
                        F.from_json(F.col("value").cast("string"), esquema_evento).alias("e"))
           .select("topic", "partition", "offset", "kafkaTimestamp", "e.*"))
pred_stream = modelo.transform(eventos)
RUTA_PRED = "./artifacts/s11_predicciones_stream"; CHK = "./artifacts/chk_s11"
JOB_ML = f"{PUSHGATEWAY}/metrics/job/lambda26_ml_streaming/instance"
publicaciones = []
def push_en_vivo(sensor, real, pred):
    cuerpo = ("# TYPE lambda26_ml_pm25_real gauge\n" f"lambda26_ml_pm25_real {real}\n"
              "# TYPE lambda26_ml_pm25_prediccion_siguiente gauge\n" f"lambda26_ml_pm25_prediccion_siguiente {pred}\n")
    urllib.request.urlopen(urllib.request.Request(f"{JOB_ML}/{sensor}", data=cuerpo.encode(), method="PUT")).close()

def por_microlote(dfm, bid):
    n = dfm.count()
    if n == 0:
        publicaciones.append(dict(batchId=bid, filas=0)); return
    dfm.select(F.lit(bid).alias("batchId"), "sensorId", "timestamp", "pm25",
               F.col("prediction").alias("pred")).write.mode("append").parquet(RUTA_PRED)
    for r in dfm.select("sensorId", "pm25", "prediction").collect():
        push_en_vivo(r["sensorId"], r["pm25"], r["prediction"])
    publicaciones.append(dict(batchId=bid, filas=n, status="guardado y publicado en vivo"))

if "q" in dir() and q.isActive: q.stop()
q = pred_stream.writeStream.foreachBatch(por_microlote).option("checkpointLocation", CHK).start()
print("consulta activa:", q.isActive)

consulta activa: True


In [8]:
serie = desde_archivo()
tail = serie[-80:-20]                      # horas reales consecutivas del histórico
h = publicar_paced(TOPIC, [aire.evento(t, a, b, "openmeteo-replay") for t, a, b in tail], pausa_s=3.0)
print(len(tail), "eventos en camino (1 cada 3 s ≈ 3 min). No detengas la consulta.")

60 eventos en camino (1 cada 3 s ≈ 3 min). No detengas la consulta.


netlib-blas: JNI_OnLoad: dlopen(libblas.so.3) failed: libblas.so.3: cannot open shared object file: No such file or directory
                                                                                

## 4. Confirmar que es en vivo: la misma consulta PromQL, dos veces (3.9)

In [9]:
q1 = prom("lambda26_ml_pm25_real"); print("Primera lectura:", q1[0]["value"][1] if q1 else None, "·", time.strftime("%H:%M:%S"))
time.sleep(20)
q2 = prom("lambda26_ml_pm25_real"); print("Segunda lectura:", q2[0]["value"][1] if q2 else None, "·", time.strftime("%H:%M:%S"))
print("¡cambió!" if q1 and q2 and q1[0]["value"][1] != q2[0]["value"][1] else "Igual: espera otros 20 s o revisa que el productor siga publicando")
print(publicaciones[-3:])

Primera lectura: 6.7 · 03:16:52
Segunda lectura: 6.7 · 03:17:12
Igual: espera otros 20 s o revisa que el productor siga publicando
[{'batchId': 58, 'filas': 1, 'status': 'guardado y publicado en vivo'}, {'batchId': 59, 'filas': 1, 'status': 'guardado y publicado en vivo'}, {'batchId': 60, 'filas': 1, 'status': 'guardado y publicado en vivo'}]


**Tablero (3.11):** importa `s11_dashboard_ecodata.json` en Grafana (Dashboards → New → Import). Pon *Last 5 minutes* y refresh *5s* mientras corre la publicación y captura el panel en vivo.

## 5. Cerrar, auditar y limpiar el valor congelado (3.8, 3.10)

In [10]:
h.join(timeout=240); time.sleep(10)
q.stop()
# Hallazgo (2.3 de la guía): con el productor detenido, el Pushgateway sigue mostrando el último valor
c1 = prom("lambda26_ml_pm25_real"); time.sleep(20); c2 = prom("lambda26_ml_pm25_real")
print("Con el productor detenido →", c1[0]["value"][1] if c1 else None, "|", c2[0]["value"][1] if c2 else None, "(idéntico = congelado, no en vivo)")

Con el productor detenido → 6.7 | 6.7 (idéntico = congelado, no en vivo)


In [11]:
res = spark.read.parquet(RUTA_PRED).orderBy("timestamp").toPandas()
res["real_siguiente"] = res["pm25"].shift(-1); aud = res.dropna()
print(f"Predicciones comparables: {len(aud)} · MAE en producción (stream): {(aud['pred'] - aud['real_siguiente']).abs().mean():.4f} µg/m³")
res.head(8)

/usr/local/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)
[Stage 317:>                                                        (0 + 2) / 2]

Predicciones comparables: 59 · MAE en producción (stream): 0.5435 µg/m³


,batchId,sensorId,timestamp,pm25,pred,real_siguiente
0,1,openmeteo-cams-juliaca,1703779200000,1.3,1.418473,1.1
1,2,openmeteo-cams-juliaca,1703782800000,1.1,1.340961,1.0
2,3,openmeteo-cams-juliaca,1703786400000,1.0,1.363362,0.9
3,4,openmeteo-cams-juliaca,1703790000000,0.9,1.385927,0.9
4,5,openmeteo-cams-juliaca,1703793600000,0.9,1.492510,0.9
5,6,openmeteo-cams-juliaca,1703797200000,0.9,1.584675,0.9
6,7,openmeteo-cams-juliaca,1703800800000,0.9,1.656142,0.9
7,8,openmeteo-cams-juliaca,1703804400000,0.9,1.702039,4.2


In [13]:
mae_pers = (aud["pm25"] - aud["real_siguiente"]).abs().mean()
print(f"MAE persistencia (mismas {len(aud)} horas): {mae_pers:.4f} µg/m³ · MAE modelo: {(aud['pred']-aud['real_siguiente']).abs().mean():.4f}")

MAE persistencia (mismas 59 horas): 0.5593 µg/m³ · MAE modelo: 0.5435


In [12]:
# Limpieza: borrar el grupo congelado para que Grafana no siga mostrando un valor viejo
sensor = res["sensorId"].iloc[0]
urllib.request.urlopen(urllib.request.Request(f"{JOB_ML}/{sensor}", method="DELETE")).close()
print("Grupo borrado del Pushgateway:", sensor)

Grupo borrado del Pushgateway: openmeteo-cams-juliaca


### Qué capturar (con **reloj del sistema y tu perfil visibles**)
1. Sección 1 (KPIs) y sección 2 (PUT 200 + verificación en Prometheus).
2. Sección 3: consulta activa y `publicaciones` con micro-lotes publicados.
3. Sección 4: las dos consultas PromQL con valores distintos y hora distinta.
4. Grafana: tablero con panel batch y panel en vivo.
5. Sección 5: valor congelado (hallazgo) y MAE en producción.

In [14]:
spark.stop()